# Lab 06 External V2 — 03 Fact Conditions

**Gold fact grain:** one row per patient-condition occurrence.

Build `fact_conditions` separately from `fact_encounters` because one encounter
can have multiple conditions.

## 1. Runtime context

In [ ]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.runtime_config import load_runtime_context


ctx = load_runtime_context(
    dbutils,
    include_validation=True,
)

config = ctx.config
run_validation = ctx.run_validation

## 2. Shared configuration

In [ ]:
from pyspark.sql import functions as F
from src.external_tables import write_external_delta

conditions_source = f"{config.reference_path}/conditions.csv"

print(f"Source : {conditions_source}")
print(f"Target : {config.fact_conditions}")
print(f"Path   : {config.table_path('fact_conditions')}")

## 3. Load source

In [ ]:
conditions_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(conditions_source)
)

print(f"Source rows    : {conditions_src.count():,}")
print(f"Source columns : {len(conditions_src.columns)}")
display(conditions_src.limit(10))

## 4. Validate source schema

In [ ]:
REQUIRED_COLUMNS = {
    "START",
    "STOP",
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "DESCRIPTION",
}

missing_columns = sorted(
    REQUIRED_COLUMNS - set(conditions_src.columns)
)

if missing_columns:
    raise ValueError(
        "conditions.csv is missing required columns: "
        + ", ".join(missing_columns)
    )

print("Condition source schema validation passed.")

## 5. Prepare condition events

Grain:

**one distinct `(patient, encounter, code, description, start, stop)` event**

In [ ]:
prepared_conditions = (
    conditions_src
    .select(
        F.to_date("START").alias("condition_start_date"),
        F.to_date("STOP").alias("condition_stop_date"),
        F.col("PATIENT").alias("patient_id"),
        F.col("ENCOUNTER").alias("encounter_id"),
        F.col("CODE").alias("condition_code"),
        F.col("DESCRIPTION").alias("condition_description"),
    )
    .dropDuplicates(
        [
            "patient_id",
            "encounter_id",
            "condition_code",
            "condition_description",
            "condition_start_date",
            "condition_stop_date",
        ]
    )
    .withColumn(
        "condition_event_key",
        F.xxhash64(
            "patient_id",
            "encounter_id",
            "condition_code",
            "condition_description",
            F.col("condition_start_date").cast("string"),
            F.coalesce(
                F.col("condition_stop_date").cast("string"),
                F.lit(""),
            ),
        ),
    )
)

display(prepared_conditions.limit(10))

## 6. Validate source/event grain

In [ ]:
prepared_count = prepared_conditions.count()

grain_profile = (
    prepared_conditions
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("condition_event_key").alias("distinct_event_keys"),
        F.sum(
            F.when(F.col("condition_event_key").isNull(), 1).otherwise(0)
        ).alias("null_event_keys"),
        F.sum(
            F.when(F.col("patient_id").isNull(), 1).otherwise(0)
        ).alias("null_patient_ids"),
        F.sum(
            F.when(F.col("condition_code").isNull(), 1).otherwise(0)
        ).alias("null_condition_codes"),
        F.sum(
            F.when(F.col("condition_start_date").isNull(), 1).otherwise(0)
        ).alias("invalid_start_dates"),
    )
    .first()
)

source_failures = []

if grain_profile["row_count"] != grain_profile["distinct_event_keys"]:
    source_failures.append("duplicate condition_event_key")
if (grain_profile["null_event_keys"] or 0) > 0:
    source_failures.append("null condition_event_key")
if (grain_profile["null_patient_ids"] or 0) > 0:
    source_failures.append("null patient_id")
if (grain_profile["null_condition_codes"] or 0) > 0:
    source_failures.append("null condition_code")
if (grain_profile["invalid_start_dates"] or 0) > 0:
    source_failures.append("invalid condition START date")

validation_df = spark.createDataFrame(
    [(
        int(grain_profile["row_count"]),
        int(grain_profile["distinct_event_keys"]),
        int(grain_profile["null_event_keys"] or 0),
        int(grain_profile["null_patient_ids"] or 0),
        int(grain_profile["null_condition_codes"] or 0),
        int(grain_profile["invalid_start_dates"] or 0),
        "PASS" if not source_failures else "FAIL",
    )],
    [
        "row_count",
        "distinct_event_keys",
        "null_event_keys",
        "null_patient_ids",
        "null_condition_codes",
        "invalid_start_dates",
        "status",
    ],
)

display(validation_df)

if run_validation and source_failures:
    raise ValueError(
        "Condition source validation failed: "
        + ", ".join(source_failures)
    )

## 7. Load Gold lookup objects

In [ ]:
dim_patient = (
    spark.table(config.dim_patient)
    .select("patient_key", "patient_id")
)

dim_condition = (
    spark.table(config.dim_condition)
    .select(
        "condition_key",
        "condition_code",
        "condition_description",
    )
)

dim_date = (
    spark.table(config.dim_date)
    .select("date_key", "full_date")
)

fact_encounter_keys = (
    spark.table(config.fact_encounters)
    .select("encounter_key", "encounter_id")
)

print("Gold lookup objects loaded.")

## 8. Resolve foreign keys

In [ ]:
fact_conditions_df = (
    prepared_conditions.alias("c")
    .join(
        dim_patient.alias("p"),
        F.col("c.patient_id") == F.col("p.patient_id"),
        "left",
    )
    .join(
        dim_condition.alias("dc"),
        (
            (F.col("c.condition_code") == F.col("dc.condition_code"))
            & (
                F.col("c.condition_description")
                == F.col("dc.condition_description")
            )
        ),
        "left",
    )
    .join(
        dim_date.alias("d"),
        F.col("c.condition_start_date") == F.col("d.full_date"),
        "left",
    )
    .join(
        fact_encounter_keys.alias("e"),
        F.col("c.encounter_id") == F.col("e.encounter_id"),
        "left",
    )
    .select(
        F.col("c.condition_event_key"),
        F.col("p.patient_key"),
        F.col("dc.condition_key"),
        F.col("d.date_key").alias("condition_start_date_key"),
        F.col("e.encounter_key"),

        F.col("c.patient_id"),
        F.col("c.encounter_id"),
        F.col("c.condition_code"),
        F.col("c.condition_description"),
        F.col("c.condition_start_date"),
        F.col("c.condition_stop_date"),

        F.when(
            F.col("c.condition_stop_date").isNull(),
            F.lit(True),
        )
        .otherwise(F.lit(False))
        .alias("is_active_condition"),

        F.when(
            F.col("c.condition_stop_date").isNotNull(),
            F.datediff(
                F.col("c.condition_stop_date"),
                F.col("c.condition_start_date"),
            ),
        )
        .otherwise(F.lit(None).cast("int"))
        .alias("condition_duration_days"),
    )
)

display(fact_conditions_df.limit(10))

## 9. Validate foreign keys

In [ ]:
fk_profile = (
    fact_conditions_df
    .agg(
        F.sum(
            F.when(F.col("patient_key").isNull(), 1).otherwise(0)
        ).alias("missing_patient_fk"),
        F.sum(
            F.when(F.col("condition_key").isNull(), 1).otherwise(0)
        ).alias("missing_condition_fk"),
        F.sum(
            F.when(
                F.col("condition_start_date_key").isNull(), 1
            ).otherwise(0)
        ).alias("missing_date_fk"),
        F.sum(
            F.when(
                F.col("encounter_id").isNotNull()
                & F.col("encounter_key").isNull(),
                1,
            ).otherwise(0)
        ).alias("unresolved_encounter_fk"),
    )
    .first()
)

fk_failures = [
    name
    for name in [
        "missing_patient_fk",
        "missing_condition_fk",
        "missing_date_fk",
        "unresolved_encounter_fk",
    ]
    if (fk_profile[name] or 0) > 0
]

fk_validation_df = spark.createDataFrame(
    [(
        int(fk_profile["missing_patient_fk"] or 0),
        int(fk_profile["missing_condition_fk"] or 0),
        int(fk_profile["missing_date_fk"] or 0),
        int(fk_profile["unresolved_encounter_fk"] or 0),
        "PASS" if not fk_failures else "FAIL",
    )],
    [
        "missing_patient_fk",
        "missing_condition_fk",
        "missing_date_fk",
        "unresolved_encounter_fk",
        "status",
    ],
)

display(fk_validation_df)

if run_validation and fk_failures:
    raise ValueError(
        "fact_conditions FK validation failed: "
        + ", ".join(fk_failures)
    )

## 10. Persist `fact_conditions`

In [ ]:
fact_path = write_external_delta(
    spark,
    fact_conditions_df,
    config,
    config.fact_conditions,
)

print(f"Created external table: {config.fact_conditions}")
print(f"Physical Delta path   : {fact_path}")

## 11. Fact grain reconciliation

In [ ]:
target_profile = (
    spark.table(config.fact_conditions)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("condition_event_key")
            .alias("distinct_event_keys"),
    )
    .first()
)

grain_status = (
    "PASS"
    if (
        target_profile["row_count"] == prepared_count
        and target_profile["row_count"]
        == target_profile["distinct_event_keys"]
    )
    else "FAIL"
)

display(
    spark.createDataFrame(
        [(
            prepared_count,
            int(target_profile["row_count"]),
            int(target_profile["distinct_event_keys"]),
            grain_status,
        )],
        [
            "prepared_rows",
            "fact_rows",
            "distinct_event_keys",
            "status",
        ],
    )
)

if run_validation and grain_status == "FAIL":
    raise ValueError(
        "fact_conditions grain reconciliation failed."
    )

## 12. Business profile

In [ ]:
condition_profile_df = (
    spark.table(config.fact_conditions)
    .groupBy(
        "condition_code",
        "condition_description",
    )
    .agg(
        F.count("*").alias("condition_events"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.sum(
            F.when(F.col("is_active_condition"), 1).otherwise(0)
        ).alias("active_condition_events"),
        F.round(
            F.avg("condition_duration_days"), 2
        ).alias("avg_duration_days"),
    )
    .orderBy(F.desc("condition_events"))
)

display(condition_profile_df.limit(25))

## 13. Final validation

In [ ]:
final_checks = [
    ("source_grain", len(source_failures) == 0),
    ("foreign_keys", len(fk_failures) == 0),
    ("fact_grain", grain_status == "PASS"),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in final_checks
    ],
    ["check_name", "status"],
)

display(final_validation_df)

failed_checks = [
    name
    for name, passed in final_checks
    if not passed
]

if run_validation and failed_checks:
    raise RuntimeError(
        "Final fact_conditions validation failed: "
        + ", ".join(failed_checks)
    )

## 14. Completion

`fact_conditions` grain:

> **One row per patient-condition occurrence.**

```text
               dim_date
                  |
                  |
dim_patient ─ fact_conditions ─ dim_condition
                  |
             fact_encounters
```

**Next:** `lab06_04_aggregations`.

In [ ]:
print("LAB 06 — FACT CONDITIONS COMPLETE")
print(f"Created: {config.fact_conditions}")
print("Next: lab06_04_aggregations")